# Unified Observability: App, GPU & LLM Monitoring + Tracing

This notebook deploys a complete observability stack that provides:

| Layer | What | Tool |
|-------|------|------|
| **App Metrics** | Request rate, latency, error rate, active orders | Prometheus + Grafana |
| **GPU Metrics** | Utilization, memory, power, temperature | DCGM Exporter + Grafana |
| **LLM Metrics** | Throughput (tok/s), TTFT, ITL, queue depth, cache | vLLM built-in + Grafana |
| **LLM Tracing** | End-to-end request traces through the inference stack | OpenTelemetry + Tempo |

### Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                      Metric Sources                             │
│  cafe-api /metrics    vLLM /metrics    DCGM Exporter           │
│       │                    │                │                   │
└───────┼────────────────────┼────────────────┼───────────────────┘
        └──────────┬─────────┘────────────────┘
                   ▼
    ┌──────────────────────────────┐
    │  OpenShift User Workload     │
    │  Monitoring (Prometheus)     │
    └──────────────┬───────────────┘
                   ▼
    ┌──────────────────────────────┐    ┌───────────────────────┐
    │       Thanos Querier         │    │ OTel Collector (CR)   │
    └──────────────┬───────────────┘    └───────────┬───────────┘
                   │                                │
                   ▼                                ▼
    ┌──────────────────────────┐    ┌───────────────────────────┐
    │   Grafana (Dashboards)   │    │  Tempo (Trace Backend)    │
    └──────────────────────────┘    └───────────┬───────────────┘
                                                │
                                                ▼
                                   ┌────────────────────────────┐
                                   │ OpenShift Console            │
                                   │ Observe → Traces             │
                                   └────────────────────────────┘
```

### Prerequisites

- Completed `0_setup/` (cluster, model, cafe-api deployed)
- `oc` CLI logged in with cluster-admin or monitoring permissions
- User Workload Monitoring enabled on the cluster
- NVIDIA GPU Operator installed (for GPU metrics)
- Observability Operators installed (Tempo, OpenTelemetry, Cluster Observability) — run `0_setup/1_environment_setup.ipynb` Section 7
- `.env` file configured with `CLUSTER_DOMAIN`, `MODEL_ENDPOINT`, `MODEL_NAME`, `MAAS_API_KEY`

## 1. Prerequisites Check

In [18]:
import subprocess, json, os, sys
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN", "")
MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", "")
MODEL_NAME = os.getenv("MODEL_NAME", "")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

if not CLUSTER_DOMAIN:
    result = subprocess.run(
        ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
        capture_output=True, text=True
    )
    CLUSTER_DOMAIN = result.stdout.strip()

print(f"Cluster Domain: {CLUSTER_DOMAIN}")
print(f"Model Namespace: {MODEL_NAMESPACE}")
print(f"Model Endpoint: {MODEL_ENDPOINT}")
print(f"Model Name: {MODEL_NAME}")
print(f"MaaS API Key: {'configured' if MAAS_API_KEY else '⚠️  not set'}")

Cluster Domain: apps.openshift-cluster.sandbox1785.opentlc.com
Model Namespace: demo
Model Endpoint: https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b
Model Name: qwen36-27b
MaaS API Key: configured


In [19]:
%%bash
echo "=== Checking prerequisites ==="
echo ""

# 1. oc login
echo -n "[1/5] oc login: "
if oc whoami &>/dev/null; then
    echo "✅ $(oc whoami)"
else
    echo "❌ Not logged in"
    exit 1
fi

# 2. User Workload Monitoring
echo -n "[2/5] User Workload Monitoring: "
UWM_PODS=$(oc get pods -n openshift-user-workload-monitoring --no-headers 2>/dev/null | wc -l)
if [ "$UWM_PODS" -gt 0 ]; then
    echo "✅ ($UWM_PODS pods running)"
else
    echo "❌ Not enabled — enable via cluster-monitoring-config ConfigMap"
fi

# 3. GPU Operator / DCGM
echo -n "[3/5] NVIDIA GPU Operator: "
GPU_NS=$(oc get ns nvidia-gpu-operator --no-headers 2>/dev/null | awk '{print $1}')
if [ -n "$GPU_NS" ]; then
    DCGM_PODS=$(oc get pods -n nvidia-gpu-operator -l app=nvidia-dcgm-exporter --no-headers 2>/dev/null | wc -l)
    echo "✅ (DCGM exporter: ${DCGM_PODS} pods)"
else
    echo "⚠️  Namespace not found — GPU metrics will be unavailable"
fi

# 4. vLLM model serving (llm-d uses app.kubernetes.io/part-of label)
echo -n "[4/5] vLLM Model Serving: "
MODEL_NS=${MODEL_NAMESPACE:-demo}
MODEL_PODS=$(oc get pods -n $MODEL_NS -l app.kubernetes.io/part-of=llminferenceservice --no-headers 2>/dev/null | grep Running | wc -l)
if [ "$MODEL_PODS" -gt 0 ]; then
    echo "✅ ($MODEL_PODS pods running in $MODEL_NS)"
else
    echo "⚠️  No running inference pods in $MODEL_NS namespace"
fi

# 5. Cafe API
echo -n "[5/5] Cafe API: "
CAFE_PODS=$(oc get pods -n cafe-system -l app=cafe-api --no-headers 2>/dev/null | grep Running | wc -l)
if [ "$CAFE_PODS" -gt 0 ]; then
    echo "✅ ($CAFE_PODS pods running)"
else
    echo "⚠️  cafe-api not running — run 0_setup/2_app_setup.ipynb first"
fi

echo ""
echo "=== Prerequisites check complete ==="

=== Checking prerequisites ===

[1/5] oc login: ✅ kube:admin
[2/5] User Workload Monitoring: ✅ (       5 pods running)
[3/5] NVIDIA GPU Operator: ✅ (DCGM exporter:        1 pods)
[4/5] vLLM Model Serving: ✅ (       2 pods running in demo)
[5/5] Cafe API: ✅ (       1 pods running)

=== Prerequisites check complete ===


## 2. Instrument Cafe App (Application Metrics + Tracing)

We add three capabilities to the cafe-order-system:
1. **Prometheus `/metrics` endpoint** — request count, latency histogram, active orders gauge
2. **OpenTelemetry tracing** — distributed traces exported to the OTel Collector
3. **LLM-instrumented `/api/recommend/` endpoint** — calls the LLM with menu context, records model/token usage as span attributes. Uses `opentelemetry-instrumentation-httpx` to auto-trace the outbound HTTP call.

The instrumented source code is already prepared in `../0_setup/apps/cafe-order-system/`. We rebuild the image and redeploy.

In [20]:
%%bash
echo "=== Rebuilding cafe-api with metrics + tracing instrumentation ==="
echo ""

cd ../0_setup

# Rebuild image
echo "Starting build..."
oc start-build cafe-api \
    --from-dir=apps/cafe-order-system \
    -n cafe-system \
    --follow --wait

echo ""
echo "Restarting deployment..."
oc rollout restart deploy/cafe-api -n cafe-system
oc wait --for=condition=available deployment/cafe-api -n cafe-system --timeout=120s

echo ""
echo "✅ cafe-api rebuilt with instrumentation"

=== Rebuilding cafe-api with metrics + tracing instrumentation ===

Starting build...


Uploading directory "apps/cafe-order-system" as binary input for the build ...

Uploading finished


build.build.openshift.io/cafe-api-12 started
Receiving source from STDIN as archive ...
time="2026-06-20T09:29:39Z" level=info msg="Not using native diff for overlay, this may cause degraded performance for building images: kernel has CONFIG_OVERLAY_FS_REDIRECT_DIR enabled"
I0620 09:29:39.863950       1 defaults.go:112] Defaulting to storage driver "overlay" with options [mountopt=metacopy=on].
Caching blobs under "/var/cache/blobs".

Pulling image python:3.11-slim ...
Resolving "python" using unqualified-search registries (/etc/containers/registries.conf)
Trying to pull registry.redhat.io/python:3.11-slim...
Trying to pull registry.access.redhat.com/python:3.11-slim...
Trying to pull quay.io/python:3.11-slim...
Trying to pull docker.io/library/python:3.11-slim...
Getting image source signatures
Copying blob sha256:3250ea7ceadc1e9a40a2d832082ee8fe68631ef9c9bf7bd5447fbcff9f527439
Copying blob sha256:72c03230f1363a3fb61d2f98504cf168bad3fe22f511ad2005dc021515d7ce97
Copying blob sha256:8c4

In [21]:
%%bash
echo "=== Verifying /metrics endpoint ==="
ROUTE=$(oc get route cafe-api -n cafe-system -o jsonpath='{.spec.host}')
echo "Route: http://${ROUTE}"
echo ""

# Send a few requests to generate metrics
curl -s "http://${ROUTE}/health" > /dev/null
curl -s "http://${ROUTE}/api/menu/" > /dev/null
curl -s "http://${ROUTE}/api/customers/" > /dev/null

echo "Prometheus metrics (sample):"
curl -s "http://${ROUTE}/metrics" | grep -E '^cafe_' | head -20

=== Verifying /metrics endpoint ===
Route: http://cafe-api-cafe-system.apps.openshift-cluster.sandbox1785.opentlc.com

Prometheus metrics (sample):
cafe_http_requests_total{endpoint="/api/customers/",method="GET",status="200"} 1.0
cafe_http_requests_created{endpoint="/api/customers/",method="GET",status="200"} 1.7819478246029222e+09
cafe_http_request_duration_seconds_bucket{endpoint="/api/customers/",le="0.005",method="GET"} 1.0
cafe_http_request_duration_seconds_bucket{endpoint="/api/customers/",le="0.01",method="GET"} 1.0
cafe_http_request_duration_seconds_bucket{endpoint="/api/customers/",le="0.025",method="GET"} 1.0
cafe_http_request_duration_seconds_bucket{endpoint="/api/customers/",le="0.05",method="GET"} 1.0
cafe_http_request_duration_seconds_bucket{endpoint="/api/customers/",le="0.1",method="GET"} 1.0
cafe_http_request_duration_seconds_bucket{endpoint="/api/customers/",le="0.25",method="GET"} 1.0
cafe_http_request_duration_seconds_bucket{endpoint="/api/customers/",le="0.5",meth

In [22]:
%%bash
echo "=== Deploying ServiceMonitor for cafe-api ==="

# Ensure the service has the correct label and port name
oc label svc/cafe-api -n cafe-system app=cafe-api --overwrite
oc patch svc cafe-api -n cafe-system --type='json' \
    -p='[{"op": "replace", "path": "/spec/ports/0/name", "value": "http"}]' 2>/dev/null || true

oc apply -f manifests/01-servicemonitor-cafe.yaml
echo ""
echo "✅ ServiceMonitor created — Prometheus will scrape cafe-api /metrics"

=== Deploying ServiceMonitor for cafe-api ===
service/cafe-api not labeled
service/cafe-api patched (no change)
servicemonitor.monitoring.coreos.com/cafe-api unchanged

✅ ServiceMonitor created — Prometheus will scrape cafe-api /metrics


## 3. LLM Metrics (vLLM)

vLLM exposes Prometheus metrics **by default** on the same port as the inference API (HTTPS on port 8000). Key metrics include:

| Metric | Description |
|--------|-------------|
| `vllm:num_requests_running` | Currently processing requests |
| `vllm:num_requests_waiting` | Queued requests |
| `vllm:avg_generation_throughput_toks_per_s` | Token generation throughput |
| `vllm:time_to_first_token_seconds` | Time to first token (TTFT) |
| `vllm:time_per_output_token_seconds` | Inter-token latency (ITL) |
| `vllm:kv_cache_usage_perc` | KV cache utilization |
| `vllm:prompt_tokens_total` | Total prompt tokens processed |
| `vllm:generation_tokens_total` | Total tokens generated |

Note: llm-d (LLMInferenceService) pods use label `app.kubernetes.io/part-of=llminferenceservice` and serve metrics over **HTTPS** with self-signed TLS.

In [23]:
%%bash
echo "=== Checking vLLM metrics availability ==="
echo ""

MODEL_NS=${MODEL_NAMESPACE:-demo}

VLLM_POD=$(oc get pods -n $MODEL_NS -l app.kubernetes.io/part-of=llminferenceservice \
    --field-selector=status.phase=Running -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -z "$VLLM_POD" ]; then
    echo "⚠️  No vLLM pods found in namespace $MODEL_NS"
    echo "   Deploy a model first via 0_setup/1_environment_setup.ipynb"
    exit 0
fi

echo "vLLM Pod: $VLLM_POD"
echo ""
echo "Sample vLLM metrics:"
oc exec -n $MODEL_NS $VLLM_POD -c kserve-container \
    -- curl -sk https://localhost:8000/metrics 2>/dev/null \
    | grep '^vllm:' | head -10 || echo "  (metrics not available yet)"

=== Checking vLLM metrics availability ===

vLLM Pod: qwen36-27b-kserve-867b69f6d5-s5kbs

Sample vLLM metrics:


In [24]:
%%bash
echo "=== Deploying PodMonitor for vLLM ==="

oc apply -f manifests/02-podmonitor-vllm.yaml
echo ""
echo "✅ PodMonitor created — Prometheus will scrape vLLM inference pods (HTTPS)"

=== Deploying PodMonitor for vLLM ===
podmonitor.monitoring.coreos.com/vllm-inference unchanged

✅ PodMonitor created — Prometheus will scrape vLLM inference pods (HTTPS)


## 4. GPU Metrics (NVIDIA DCGM)

The NVIDIA GPU Operator deploys the **DCGM Exporter** as a DaemonSet on GPU nodes. It exposes metrics such as:

| Metric | Description |
|--------|-------------|
| `DCGM_FI_DEV_GPU_UTIL` | GPU utilization (%) |
| `DCGM_FI_DEV_FB_USED` | Frame buffer memory used (MiB) |
| `DCGM_FI_DEV_FB_FREE` | Frame buffer memory free (MiB) |
| `DCGM_FI_DEV_POWER_USAGE` | Power draw (W) |
| `DCGM_FI_DEV_GPU_TEMP` | Temperature (C) |
| `DCGM_FI_DEV_SM_CLOCK` | Streaming Multiprocessor clock (MHz) |

These are typically already scraped by the platform Prometheus via the GPU Operator's ServiceMonitor.

In [25]:
%%bash
echo "=== Verifying DCGM Exporter ==="
echo ""

GPU_NS="nvidia-gpu-operator"
if ! oc get ns $GPU_NS &>/dev/null; then
    echo "⚠️  Namespace $GPU_NS not found"
    GPU_NS=$(oc get pods --all-namespaces -l app=nvidia-dcgm-exporter -o jsonpath='{.items[0].metadata.namespace}' 2>/dev/null)
    if [ -z "$GPU_NS" ]; then
        echo "❌ DCGM Exporter not found on this cluster"
        echo "   GPU metrics will not be available in Grafana"
        exit 0
    fi
fi

echo "GPU Operator namespace: $GPU_NS"
echo ""
echo "DCGM Exporter pods:"
oc get pods -n $GPU_NS -l app=nvidia-dcgm-exporter --no-headers 2>/dev/null || \
    oc get pods -n $GPU_NS -l app.kubernetes.io/name=dcgm-exporter --no-headers 2>/dev/null || \
    echo "  (pods not found with expected labels)"

echo ""
echo "Checking if DCGM metrics are already scraped..."
SM=$(oc get servicemonitor -n $GPU_NS -o name 2>/dev/null | grep -i dcgm | head -1)
if [ -n "$SM" ]; then
    echo "✅ ServiceMonitor already exists: $SM"
    echo "   GPU metrics are being collected by Prometheus"
else
    echo "⚠️  No DCGM ServiceMonitor found — GPU Operator usually configures this automatically"
fi

=== Verifying DCGM Exporter ===

GPU Operator namespace: nvidia-gpu-operator

DCGM Exporter pods:
nvidia-dcgm-exporter-7v45c   1/1   Running   2 (6h11m ago)   6h13m

Checking if DCGM metrics are already scraped...
✅ ServiceMonitor already exists: servicemonitor.monitoring.coreos.com/nvidia-dcgm-exporter
   GPU metrics are being collected by Prometheus


## 5. LLM Call Tracing (OpenTelemetry + Tempo)

Deploy the distributed tracing stack using Red Hat supported operators (installed in `0_setup/`):
1. **TempoMonolithic** — trace backend with multi-tenancy gateway (`mode: openshift`)
2. **RBAC** — ClusterRoles for trace read/write via the Tempo gateway
3. **OpenTelemetryCollector** — receives OTLP traces and forwards to Tempo gateway
4. **UIPlugin** — enables OpenShift Console Observe → Traces

The cafe-api app is already instrumented with OpenTelemetry (via the rebuild in Section 2). We deploy the trace backend and collector, then point cafe-api at the collector.

In [26]:
%%bash
echo "=== Checking tracing operator prerequisites ==="
echo "   (Install via 0_setup/1_environment_setup.ipynb Section 7 if missing)"
echo ""

FAIL=false
for CHECK in \
    "tempo|openshift-tempo-operator|Tempo Operator" \
    "opentelemetry|openshift-opentelemetry-operator|OpenTelemetry Operator" \
    "cluster-observability-operator|openshift-operators|Cluster Observability Operator"; do
    IFS='|' read -r GREP NS LABEL <<< "$CHECK"
    CSV=$(oc get csv -n "$NS" --no-headers 2>/dev/null | grep "$GREP" | grep Succeeded | awk '{print $1}' | head -1)
    if [ -n "$CSV" ]; then
        echo "✅ $LABEL: $CSV"
    else
        echo "❌ $LABEL: not found — run 0_setup/1_environment_setup.ipynb first"
        FAIL=true
    fi
done

if $FAIL; then exit 1; fi

echo ""
echo "=== Creating monitoring namespace ==="
oc apply -f manifests/00-namespace.yaml

echo ""
echo "=== Deploying TempoMonolithic (multi-tenancy gateway + tenant 'dev') ==="
oc apply -f manifests/04-tempo.yaml

echo ""
echo "=== Applying trace read/write RBAC ==="
oc apply -f manifests/04b-tempo-rbac.yaml

echo ""
echo "=== Deploying OpenTelemetryCollector CR ==="
oc apply -f manifests/03-otel-collector.yaml

echo ""
echo "=== Enabling distributed tracing console plugin ==="
oc apply -f manifests/04a-uiplugin-tracing.yaml

echo ""
echo "Waiting for Tempo (StatefulSet)..."
oc wait --for=jsonpath='{.status.readyReplicas}'=1 statefulset/tempo-lab -n monitoring --timeout=180s 2>/dev/null || \
    echo "⚠️  Tempo not ready yet — run: oc get tempomonolithic lab -n monitoring -o yaml"

echo "Waiting for Tempo gateway..."
oc wait --for=condition=available deployment/tempo-lab-gateway -n monitoring --timeout=120s 2>/dev/null || \
    echo "⚠️  Gateway not ready yet — run: oc get deployment tempo-lab-gateway -n monitoring"

echo "Waiting for OTel Collector..."
oc wait --for=condition=available deployment/otel-collector -n monitoring --timeout=120s 2>/dev/null || \
    echo "⚠️  OTel Collector not ready yet — run: oc get opentelemetrycollector otel -n monitoring -o yaml"

echo ""
echo "✅ Tracing stack deployed (Tempo + Gateway + OTel Collector + Console Plugin)"

=== Checking tracing operator prerequisites ===
   (Install via 0_setup/1_environment_setup.ipynb Section 7 if missing)

✅ Tempo Operator: tempo-operator.v0.21.0-1
✅ OpenTelemetry Operator: opentelemetry-operator.v0.152.0-1
✅ Cluster Observability Operator: cluster-observability-operator.v1.5.0

=== Creating monitoring namespace ===
namespace/monitoring unchanged

=== Deploying TempoMonolithic (multi-tenancy gateway + tenant 'dev') ===
tempomonolithic.tempo.grafana.com/lab unchanged

=== Applying trace read/write RBAC ===
clusterrole.rbac.authorization.k8s.io/tempomonolithic-traces-reader unchanged
clusterrolebinding.rbac.authorization.k8s.io/tempomonolithic-traces-reader unchanged
clusterrole.rbac.authorization.k8s.io/tempomonolithic-traces-writer unchanged
clusterrolebinding.rbac.authorization.k8s.io/tempomonolithic-traces-writer unchanged

=== Deploying OpenTelemetryCollector CR ===
opentelemetrycollector.opentelemetry.io/otel unchanged

=== Enabling distributed tracing console plug

In [27]:
import os, subprocess
from dotenv import load_dotenv

load_dotenv("../.env")

MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", "")
MODEL_NAME = os.getenv("MODEL_NAME", "")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

print("=== Configuring cafe-api for tracing + LLM access ===")

env_args = [
    "oc", "set", "env", "deploy/cafe-api", "-n", "cafe-system",
    "OTEL_EXPORTER_OTLP_ENDPOINT=otel-collector.monitoring.svc.cluster.local:4317",
]

if MODEL_ENDPOINT:
    env_args.append(f"MODEL_ENDPOINT={MODEL_ENDPOINT}")
    print(f"  MODEL_ENDPOINT={MODEL_ENDPOINT}")
if MODEL_NAME:
    env_args.append(f"MODEL_NAME={MODEL_NAME}")
    print(f"  MODEL_NAME={MODEL_NAME}")
if MAAS_API_KEY:
    env_args.append(f"MAAS_API_KEY={MAAS_API_KEY}")
    print(f"  MAAS_API_KEY={MAAS_API_KEY[:12]}...")

subprocess.run(env_args, check=True)

r = subprocess.run(
    ["oc", "rollout", "status", "deploy/cafe-api", "-n", "cafe-system", "--timeout=60s"],
    capture_output=True, text=True,
)
print(r.stdout.strip())

print()
print("✅ cafe-api configured (OTel tracing + LLM recommend endpoint)")

=== Configuring cafe-api for tracing + LLM access ===
  MODEL_ENDPOINT=https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b
  MODEL_NAME=qwen36-27b
  MAAS_API_KEY=sk-oai-1ILU1...
deployment.apps/cafe-api updated
Waiting for deployment "cafe-api" rollout to finish: 1 old replicas are pending termination...
Waiting for deployment "cafe-api" rollout to finish: 1 old replicas are pending termination...
deployment "cafe-api" successfully rolled out

✅ cafe-api configured (OTel tracing + LLM recommend endpoint)


### Enable vLLM Built-in Tracing (Optional)

vLLM includes OpenTelemetry support via the `--otlp-traces-endpoint` CLI argument. When enabled, it exports internal inference spans (token generation, scheduling, etc.) to the OTel Collector. This gives visibility into what happens **inside** the model server, complementing the application-level traces from cafe-api.

> **Note:** vLLM requires a **CLI argument** (`--otlp-traces-endpoint`), not just environment variables. This cell appends the flag to `VLLM_ADDITIONAL_ARGS`, which triggers a pod restart. GPU scheduling may take a few minutes.

In [28]:
import os, subprocess, json
from dotenv import load_dotenv

load_dotenv("../.env")

MODEL_NAME = os.getenv("MODEL_NAME", "")
MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
OTEL_GRPC_ENDPOINT = "http://otel-collector.monitoring.svc.cluster.local:4317"

if not MODEL_NAME:
    print("⚠️  MODEL_NAME not set in .env — skipping vLLM OTel setup")
else:
    r = subprocess.run(
        ["oc", "get", "llminferenceservice", MODEL_NAME, "-n", MODEL_NAMESPACE,
         "-o", "jsonpath={.spec.template.containers[0].env[?(@.name=='VLLM_ADDITIONAL_ARGS')].value}"],
        capture_output=True, text=True,
    )
    current_args = r.stdout.strip()

    if "--otlp-traces-endpoint" in current_args:
        print(f"✅ vLLM OTel tracing already configured for {MODEL_NAME}")
        print(f"   VLLM_ADDITIONAL_ARGS: {current_args}")
    else:
        new_args = f"{current_args} --otlp-traces-endpoint={OTEL_GRPC_ENDPOINT}".strip()
        print(f"=== Enabling OTel tracing on vLLM ({MODEL_NAME}) ===")
        print(f"   VLLM_ADDITIONAL_ARGS: {new_args}")

        patch = json.dumps([
            {"op": "replace",
             "path": "/spec/template/containers/0/env",
             "value": [
                 {"name": "VLLM_ADDITIONAL_ARGS", "value": new_args},
                 {"name": "OTEL_SERVICE_NAME", "value": f"vllm-{MODEL_NAME}"},
             ]},
        ])
        subprocess.run(
            ["oc", "patch", "llminferenceservice", MODEL_NAME, "-n", MODEL_NAMESPACE,
             "--type=json", "-p", patch],
            check=True,
        )
        print()
        print("✅ vLLM OTel tracing enabled — pod will restart (GPU scheduling may take a few minutes)")
        print("   Traces will appear under the vLLM service name in Observe → Traces")

✅ vLLM OTel tracing already configured for qwen36-27b


## 6. Grafana Dashboards (Unified View)

Deploy Grafana with pre-configured datasources and dashboards:
- **Prometheus** datasource → queries Thanos Querier for app/GPU/LLM metrics
- **Tempo** datasource → correlates traces with metrics
- Pre-built dashboards: App, GPU, LLM

In [29]:
%%bash
echo "=== Creating Grafana dashboard ConfigMaps ==="

oc create configmap grafana-dashboard-app \
    --from-file=app-dashboard.json=manifests/07-dashboard-app.json \
    -n monitoring --dry-run=client -o yaml | oc apply -f -

oc create configmap grafana-dashboard-gpu \
    --from-file=gpu-dashboard.json=manifests/08-dashboard-gpu.json \
    -n monitoring --dry-run=client -o yaml | oc apply -f -

oc create configmap grafana-dashboard-llm \
    --from-file=llm-dashboard.json=manifests/09-dashboard-llm.json \
    -n monitoring --dry-run=client -o yaml | oc apply -f -

echo "✅ Dashboard ConfigMaps created"

=== Creating Grafana dashboard ConfigMaps ===
configmap/grafana-dashboard-app unchanged
configmap/grafana-dashboard-gpu unchanged
configmap/grafana-dashboard-llm unchanged
✅ Dashboard ConfigMaps created


In [30]:
%%bash
echo "=== Deploying Grafana ==="

# Deploy Grafana (includes ServiceAccount, Deployment, Service, Route)
oc apply -f manifests/05-grafana.yaml

# Grant monitoring view permission to grafana SA
oc adm policy add-cluster-role-to-user cluster-monitoring-view \
    -z grafana -n monitoring 2>/dev/null || true

# Create SA token for Prometheus auth
GRAFANA_TOKEN=$(oc create token grafana -n monitoring --duration=8760h 2>/dev/null || echo "")

# Apply datasources config
oc apply -f manifests/06-grafana-datasources.yaml

if [ -n "$GRAFANA_TOKEN" ]; then
    # Patch datasource configmap with real token
    oc get cm grafana-datasources -n monitoring -o yaml | \
        sed "s|\${GRAFANA_SA_TOKEN}|$GRAFANA_TOKEN|" | \
        oc apply -f -
    echo "✅ Grafana SA token configured for Prometheus access"
else
    echo "⚠️  Could not create token — Grafana may need manual Prometheus auth"
fi

# Wait and restart to pick up all configmaps
oc wait --for=condition=available deployment/grafana -n monitoring --timeout=120s
oc rollout restart deploy/grafana -n monitoring
oc wait --for=condition=available deployment/grafana -n monitoring --timeout=120s

echo ""
echo "✅ Grafana deployed and configured"

=== Deploying Grafana ===
deployment.apps/grafana unchanged
serviceaccount/grafana unchanged
service/grafana unchanged
route.route.openshift.io/grafana unchanged
clusterrole.rbac.authorization.k8s.io/cluster-monitoring-view added: "grafana"
configmap/grafana-datasources configured
configmap/grafana-dashboards-config unchanged
configmap/grafana-datasources configured
✅ Grafana SA token configured for Prometheus access
deployment.apps/grafana condition met
deployment.apps/grafana restarted
deployment.apps/grafana condition met

✅ Grafana deployed and configured


## 7. Verification & Demo

Generate traffic (both app and LLM requests) and verify all metrics/traces are flowing.

The test traffic cell calls the `/api/recommend/` endpoint with different moods, which triggers an end-to-end traced LLM call: **cafe-api** → **vLLM** (via MaaS Gateway). You can then view the resulting traces in OpenShift Console → Observe → Traces, searching for service `cafe-order-system`.

In [31]:
import subprocess, json, urllib.parse

route_r = subprocess.run(
    ["oc", "get", "route", "cafe-api", "-n", "cafe-system", "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True,
)
ROUTE = route_r.stdout.strip()
print("=== Generating test traffic (Cafe API) ===")

for i in range(10):
    subprocess.run(["curl", "-s", f"http://{ROUTE}/api/menu/"], capture_output=True)
    subprocess.run(["curl", "-s", f"http://{ROUTE}/api/customers/"], capture_output=True)
    subprocess.run(["curl", "-s", "-X", "POST", f"http://{ROUTE}/api/orders/",
                     "-H", "Content-Type: application/json",
                     "-d", '{"customer_id":1,"items":[{"menu_item_id":1,"quantity":1}]}'],
                    capture_output=True)

print("✅ Sent 10 rounds of CRUD requests")
print()

# LLM Recommend — end-to-end traced call: cafe-api → LLM
print("=== Testing LLM Recommend endpoint (traced) ===")
moods = ["something sweet", "a refreshing iced drink", "warm and cozy", "need caffeine", "anything light"]
for mood in moods:
    url = f"http://{ROUTE}/api/recommend/?mood={urllib.parse.quote(mood)}"
    r = subprocess.run(["curl", "-s", url], capture_output=True, text=True, timeout=90)
    try:
        d = json.loads(r.stdout)
        print(f"  ✅ [{d['model']}] {d['recommendation'][:60]}...")
    except Exception:
        body = r.stdout[:80] if r.stdout else "(empty)"
        print(f"  ⚠️  mood='{mood}' — {body}")

print()
print("✅ LLM recommend traffic generated (check Observe → Traces for llm.* span attributes)")

=== Generating test traffic (Cafe API) ===
✅ Sent 10 rounds of CRUD requests

=== Testing LLM Recommend endpoint (traced) ===
  ✅ [qwen36-27b] Here's a thinking process:

1.  **Analyze User Input:**
   -...
  ✅ [qwen36-27b] Here's a thinking process:

1.  **Analyze User Input:**
   -...
  ✅ [qwen36-27b] Here's a thinking process:

1.  **Analyze User Input:**
   -...
  ✅ [qwen36-27b] Here's a thinking process:

1.  **Analyze User Input:**
   -...
  ✅ [qwen36-27b] Here's a thinking process:

1.  **Analyze User Input:**
   -...

✅ LLM recommend traffic generated (check Observe → Traces for llm.* span attributes)


In [32]:
import os, json, subprocess
from dotenv import load_dotenv

load_dotenv("../.env")

MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", "")
MODEL_NAME = os.getenv("MODEL_NAME", "")
MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

print("=== Generating LLM traffic (for vLLM metrics) ===")
print()

success = False

# Method 1: Try MaaS gateway endpoint
if MODEL_ENDPOINT and MAAS_API_KEY:
    import httpx
    print(f"[Method 1] MaaS Gateway: {MODEL_ENDPOINT}")
    try:
        resp = httpx.post(
            f"{MODEL_ENDPOINT}/v1/chat/completions",
            headers={"Authorization": f"Bearer {MAAS_API_KEY}"},
            json={
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": "Say hello in Korean."}],
                "max_tokens": 50
            },
            timeout=50,
            verify=False
        )
        if resp.status_code == 200:
            content = resp.json()["choices"][0]["message"]["content"][:60]
            print(f"  ✅ Response: {content}...")
            success = True
        else:
            print(f"  ⚠️  HTTP {resp.status_code} — trying in-cluster fallback")
    except Exception as e:
        print(f"  ⚠️  {e} — trying in-cluster fallback")
    print()

# Method 2: In-cluster direct call via oc exec
if not success:
    print(f"[Method 2] In-cluster direct call to vLLM workload service")
    svc_result = subprocess.run(
        ["oc", "get", "svc", "-n", MODEL_NAMESPACE,
         "-l", "app.kubernetes.io/component=llminferenceservice-workload",
         "-o", "jsonpath={.items[0].metadata.name}"],
        capture_output=True, text=True
    )
    svc_name = svc_result.stdout.strip()
    if svc_name:
        cmd = (
            f'curl -sk -X POST https://{svc_name}.{MODEL_NAMESPACE}.svc.cluster.local:8000/v1/chat/completions '
            f'-H "Content-Type: application/json" '
            f'-d \'{{"model":"{MODEL_NAME}","messages":[{{"role":"user","content":"Hi"}}],"max_tokens":20}}\''
        )
        for i in range(3):
            result = subprocess.run(
                ["oc", "run", f"llm-test-{i}", "--rm", "-i", "--restart=Never",
                 "--image=curlimages/curl:latest", "-n", MODEL_NAMESPACE,
                 "--", "sh", "-c", cmd],
                capture_output=True, text=True, timeout=30
            )
            if "choices" in result.stdout:
                try:
                    data = json.loads(result.stdout)
                    content = data["choices"][0]["message"]["content"][:60]
                    print(f"  [{i+1}/3] ✅ {content}...")
                    success = True
                except:
                    print(f"  [{i+1}/3] ✅ Got response")
                    success = True
            else:
                print(f"  [{i+1}/3] ⚠️  {result.stdout[:80] or result.stderr[:80]}")
    else:
        print("  ⚠️  No workload service found")

print()
if success:
    print("✅ LLM traffic generated — vLLM metrics (TTFT, throughput, etc.) will update")
else:
    print("⚠️  Could not generate LLM traffic via API")
    print("   vLLM metrics are still collected — use IDE/other clients to trigger inference")
    print("   Gauge metrics (cache usage, running requests) are available regardless")

=== Generating LLM traffic (for vLLM metrics) ===

[Method 1] MaaS Gateway: https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b
  ✅ Response: Here's a thinking process:

1.  **Analyze User Input:**
   -...


✅ LLM traffic generated — vLLM metrics (TTFT, throughput, etc.) will update


In [33]:
import subprocess, json, os, urllib.request, urllib.parse, ssl
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN", "")
MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
                      capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

THANOS_URL = f"https://thanos-querier-openshift-monitoring.{CLUSTER_DOMAIN}"
TOKEN = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True).stdout.strip()

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

def query_prometheus(promql: str) -> dict:
    url = f"{THANOS_URL}/api/v1/query?query={urllib.parse.quote(promql)}"
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())

print("=== Prometheus Metric Verification ===")
print()

ns = MODEL_NAMESPACE
checks = [
    ("Cafe App — request count", f'sum(cafe_http_requests_total{{namespace="cafe-system"}})'),
    ("Cafe App — avg latency (ms)", f'avg(rate(cafe_http_request_duration_seconds_sum{{namespace="cafe-system"}}[5m]) / rate(cafe_http_request_duration_seconds_count{{namespace="cafe-system"}}[5m])) * 1000'),
    ("vLLM — running requests", f'vllm:num_requests_running{{namespace="{ns}"}}'),
    ("vLLM — KV cache usage (%)", f'vllm:kv_cache_usage_perc{{namespace="{ns}"}} * 100'),
    ("GPU — utilization (%)", 'DCGM_FI_DEV_GPU_UTIL'),
    ("GPU — memory used (MiB)", 'DCGM_FI_DEV_FB_USED'),
]

for label, promql in checks:
    try:
        result = query_prometheus(promql)
        data = result.get("data", {}).get("result", [])
        if data:
            value = data[0].get("value", ["", "N/A"])[1]
            try:
                value = f"{float(value):.2f}"
            except (ValueError, TypeError):
                pass
            print(f"  ✅ {label}: {value}")
        else:
            print(f"  ⚠️  {label}: no data yet (may take 1-2 min for first scrape)")
    except Exception as e:
        print(f"  ❌ {label}: {e}")

=== Prometheus Metric Verification ===

  ✅ Cafe App — request count: 35.00
  ✅ Cafe App — avg latency (ms): nan
  ✅ vLLM — running requests: 1.00
  ✅ vLLM — KV cache usage (%): 2.04
  ✅ GPU — utilization (%): 62.00
  ✅ GPU — memory used (MiB): 40486.00


In [34]:
import subprocess, os
from dotenv import load_dotenv

load_dotenv("../.env")
CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN", "")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
                      capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

grafana_host = subprocess.run(
    ["oc", "get", "route", "grafana", "-n", "monitoring", "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True
).stdout.strip()

console_host = subprocess.run(
    ["oc", "get", "route", "console", "-n", "openshift-console", "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True
).stdout.strip()

cafe_host = subprocess.run(
    ["oc", "get", "route", "cafe-api", "-n", "cafe-system", "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True
).stdout.strip()

print("="*70)
print("  MONITORING STACK — ACCESS INFORMATION")
print("="*70)
print()
print("-"*70)
print("  Cafe App")
print("-"*70)
print(f"    • Swagger UI:   http://{cafe_host}/docs")
print(f"    • API Base:     http://{cafe_host}/api")
print(f"    • LLM Recommend: http://{cafe_host}/api/recommend/?mood=something+sweet")
print(f"    • Metrics:      http://{cafe_host}/metrics")
print()
print("-"*70)
print("  Grafana Dashboards (admin / admin)")
print("-"*70)
print(f"    • App:  https://{grafana_host}/d/cafe-app-metrics")
print(f"    • GPU:  https://{grafana_host}/d/gpu-dcgm-metrics")
print(f"    • LLM:  https://{grafana_host}/d/llm-vllm-metrics")
print()
print("-"*70)
print("  Distributed Tracing (OpenShift Console)")
print("-"*70)
print(f"    • Observe → Traces: https://{console_host}/observe/traces")
print(f"    • Select namespace: monitoring")
print(f"    • Services:  cafe-order-system  (app traces)")
print(f"    •            vllm-*             (LLM inference traces, if GPU available)")
print(f"    • Look for:  llm.request.model, llm.usage.* span attributes")
print()
print("="*70)

  MONITORING STACK — ACCESS INFORMATION

----------------------------------------------------------------------
  Cafe App
----------------------------------------------------------------------
    • Swagger UI:   http://cafe-api-cafe-system.apps.openshift-cluster.sandbox1785.opentlc.com/docs
    • API Base:     http://cafe-api-cafe-system.apps.openshift-cluster.sandbox1785.opentlc.com/api
    • LLM Recommend: http://cafe-api-cafe-system.apps.openshift-cluster.sandbox1785.opentlc.com/api/recommend/?mood=something+sweet
    • Metrics:      http://cafe-api-cafe-system.apps.openshift-cluster.sandbox1785.opentlc.com/metrics

----------------------------------------------------------------------
  Grafana Dashboards (admin / admin)
----------------------------------------------------------------------
    • App:  https://grafana-monitoring.apps.openshift-cluster.sandbox1785.opentlc.com/d/cafe-app-metrics
    • GPU:  https://grafana-monitoring.apps.openshift-cluster.sandbox1785.opentlc.com/d

## 8. Summary

The complete observability stack is now deployed:

| Component | Namespace | Purpose |
|-----------|-----------|----------|
| ServiceMonitor (cafe-api) | `cafe-system` | Scrape app metrics |
| PodMonitor (vLLM) | `demo` | Scrape LLM inference metrics (HTTPS) |
| DCGM Exporter | `nvidia-gpu-operator` | GPU metrics (pre-existing) |
| TempoMonolithic | `monitoring` | Trace storage backend (Tempo Operator) |
| OpenTelemetryCollector | `monitoring` | Receive & forward traces (OTel Operator) |
| Grafana | `monitoring` | Unified metric dashboards |
| vLLM OTel (optional) | `demo` | Internal inference tracing via OTLP |

### Tracing Coverage

| Service Name | What's Traced |
|-------------|---------------|
| `cafe-order-system` | All FastAPI endpoints, including `/api/recommend/` → LLM calls |
| `vllm-<model>` | Internal inference spans (scheduling, token generation) — requires GPU |

The `/api/recommend/` endpoint is instrumented end-to-end:
- `cafe-order-system` span with `llm.request.model`, `llm.usage.*` attributes
- Child `POST /v1/chat/completions` span auto-traced by `httpx` instrumentation

### What You Can See

| View | Key Panels |
|------|------------|
| **Grafana — Cafe App** | Request rate, error rate, latency percentiles, active orders |
| **Grafana — GPU** | Utilization %, memory used/free, power, temperature, clocks |
| **Grafana — LLM** | Running/waiting requests, tok/s throughput, TTFT, ITL, KV cache usage |
| **OpenShift Console — Observe → Traces** | Request traces, LLM call spans, span attributes |

### Cleanup

```bash
# Remove monitoring stack
oc delete namespace monitoring
oc delete servicemonitor cafe-api -n cafe-system
oc delete podmonitor vllm-inference -n demo

# Remove OTel/LLM env from cafe-api
oc set env deploy/cafe-api -n cafe-system OTEL_EXPORTER_OTLP_ENDPOINT- MODEL_ENDPOINT- MODEL_NAME- MAAS_API_KEY-
```

### Next Steps

- Add alerting rules (PrometheusRule CRs) for SLO violations
- Integrate with MaaS gateway metrics (Limitador/Authorino) from `4_control/`
- Enable log correlation with Loki + Grafana